In [7]:
from numpy import asarray
from pandas import read_csv
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# 1. Đọc file, chỉ lấy 8 cột dưới đây, không lấy date
filename = '/content/LSTM-Multivariate_pollution.csv'
all_attributes = [
    'pollution','dew', 'temp', 'press', 'wnd_dir',
    'wnd_spd', 'snow', 'rain'
]
df = read_csv(
    filename, usecols=all_attributes, encoding='utf-8'
)[all_attributes]

# 2. Chỉ chuyển hướng gió thành số
le = LabelEncoder()
df['wnd_dir'] = le.fit_transform(df['wnd_dir'])

# 3. Lấy các giá trị và lưu vào biến data
data = df.values

print(data)

n = len(data)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = data[:train_end]
val = data[train_end:val_end]
test = data[val_end:]


print("Train:", train.shape)
print("Val:", val.shape)
print("Test:", test.shape)


[[129.   -16.    -4.   ...   1.79   0.     0.  ]
 [148.   -15.    -4.   ...   2.68   0.     0.  ]
 [159.   -11.    -5.   ...   3.57   0.     0.  ]
 ...
 [ 10.   -22.    -3.   ... 242.7    0.     0.  ]
 [  8.   -22.    -4.   ... 246.72   0.     0.  ]
 [ 12.   -21.    -3.   ... 249.85   0.     0.  ]]
Train: (30659, 8)
Val: (6571, 8)
Test: (6570, 8)


In [10]:
def split_sequences(data, n_steps, n_pred, pred_col):
    X, y = [], []
    for i in range(len(data) - n_steps - n_pred + 1):
        X.append(data[i:i+n_steps,:])
        y.append(data[i+n_steps:i+n_steps+n_pred, pred_col])
    return asarray(X), asarray(y)

n_steps = 3
n_pred = 2
pred_col = 0

X_train, y_train = split_sequences(train, n_steps, n_pred, pred_col)
X_val, y_val = split_sequences(val, n_steps, n_pred, pred_col)
X_test, y_test = split_sequences(test, n_steps, n_pred, pred_col)

print(X_train[0])
print(y_train[0])

# print(X_train[-1])
# print(y_train[-1])

[[ 129.    -16.     -4.   1020.      2.      1.79    0.      0.  ]
 [ 148.    -15.     -4.   1020.      2.      2.68    0.      0.  ]
 [ 159.    -11.     -5.   1021.      2.      3.57    0.      0.  ]]
[181. 138.]


In [11]:
y_train = y_train.reshape(y_train.shape[0], n_pred, 1)
y_val = y_val.reshape(y_val.shape[0], n_pred, 1)
y_test = y_test.reshape(y_test.shape[0], n_pred, 1)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)


X_train: (30655, 3, 8)
y_train: (30655, 2, 1)


In [13]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, RepeatVector, TimeDistributed
from sklearn.metrics import r2_score
from numpy import sqrt

model = Sequential()

# Encoder
model.add(SimpleRNN(100, activation='relu',
                    input_shape=(X_train.shape[1], X_train.shape[2])))

# Repeat context vector
model.add(RepeatVector(n_pred))

# Decoder
model.add(SimpleRNN(100, activation='relu', return_sequences=True))

# Output
model.add(TimeDistributed(Dense(1)))

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val, y_val)
)

mse, mae = model.evaluate(X_test, y_test, verbose=0)
print('MSE: %.3f, RMSE: %.3f, MAE: %.3f' % (mse, sqrt(mse), mae))

y_pred = model.predict(X_test)
print("Du bao dau tien:", y_pred[0].reshape(-1))
print("Thuc te dau tien:", y_test[0].reshape(-1))
print('R2: ', r2_score(y_test.reshape(-1), y_pred.reshape(-1)))


/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_2 (SimpleRNN)        │ (None, 100)            │        10,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_1 (RepeatVector)  │ (None, 2, 100)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (None, 2, 100)         │        20,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 2, 1)           │           101 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,101 (121.49 KB)

 Trainable params: 31,101 (121.49 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
958/958 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - loss: 1565.0901 - mae: 22.8859 - val_loss: 1026.5002 - val_mae: 18.9955
Epoch 2/20
958/958 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 1233.4911 - mae: 20.1248 - val_loss: 1015.8734 - val_mae: 18.4878
Epoch 3/20
958/958 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 1170.8666 - mae: 19.1748 - val_loss: 1030.4568 - val_mae: 19.4839
Epoch 4/20
958/958 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - loss: 1143.2321 - mae: 18.8041 - val_loss: 990.9384 - val_mae: 18.8899
Epoch 5/20
958/958 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 1133.8658 - mae: 18.6605 - val_loss: 955.6672 - val_mae: 18.7032
Epoch 6/20
958/958 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 1121.5223 - mae: 18.5994 - val_loss: 893.0450 - val_mae: 16.5556
Epoch 7/20
958/958 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 1110.3081 - mae: 18.3212 - val_loss: 1052.6025 - val_mae: 21.9656
Epoch 8/20
958/958 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 1096.4142 - mae: 18.0449 - val_loss: 958.4154 - val_mae: 17.8